In [ ]:
import optuna
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def objective_rf(trial):

    param = {

        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
        

        'max_depth': trial.suggest_categorical('max_depth', [None, 10, 20, 30, 40, 50]),
        

        'min_samples_split': trial.suggest_int('min_samples_split', 2, 15),
        

        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
        

        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None])
    }


    model = RandomForestRegressor(
        **param, 
        random_state=42, 
        n_jobs=-1 
    )


    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
    
    return -scores.mean()


study_rf = optuna.create_study(direction='minimize')
print("Starting Optuna optimization for Random Forest...")

study_rf.optimize(objective_rf, n_trials=50)

print("\nBest Trial:")
print("  Value (Log MAE):", study_rf.best_trial.value)
print("  Params: ")
for key, value in study_rf.best_trial.params.items():
    print(f"    {key}: {value}")
print("\nTraining final Random Forest model with best parameters...")
best_rf_model = RandomForestRegressor(
    **study_rf.best_trial.params, 
    random_state=42, 
    n_jobs=-1
)
best_rf_model.fit(X_train, y_train)
y_pred_log_rf = best_rf_model.predict(X_test)
y_pred_real_rf = np.expm1(y_pred_log_rf)
y_test_real = np.expm1(y_test)
mae_rf = mean_absolute_error(y_test_real, y_pred_real_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test_real, y_pred_real_rf))
r2_rf = r2_score(y_test_real, y_pred_real_rf)

print("\n--- Final RandomForest Model Evaluation on Test Data ---")
print(f"MAE (Real Price): {mae_rf:,.0f}")
print(f"RMSE (Real Price): {rmse_rf:,.0f}")
print(f"R2 Score: {r2_rf:.5f}")


In [ ]:
import optuna
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def objective_rf(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
        'max_depth': trial.suggest_int('max_depth', 10, 50),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None])
    }

    model = RandomForestRegressor(**param, random_state=42, n_jobs=-1)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
    return -scores.mean()

study_rf = optuna.create_study(direction='minimize')
print("Starting Optuna optimization for Random Forest...")
study_rf.optimize(objective_rf, n_trials=50)

print("\nBest Trial:")
print("  Value (Log MAE):", study_rf.best_trial.value)
print("  Params: ")
for key, value in study_rf.best_trial.params.items():
    print(f"    {key}: {value}")
print("\nTraining final Random Forest model with best parameters...")
best_rf_model = RandomForestRegressor(**study_rf.best_trial.params, random_state=42, n_jobs=-1)
best_rf_model.fit(X_train, y_train)
y_pred_log_rf = best_rf_model.predict(X_test)

y_pred_real_rf = np.expm1(y_pred_log_rf)
y_test_real = np.expm1(y_test)

mae_rf = mean_absolute_error(y_test_real, y_pred_real_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test_real, y_pred_real_rf))
r2_rf = r2_score(y_test_real, y_pred_real_rf)

print("\n--- Final RandomForest Model Evaluation on Test Data ---")
print(f"MAE (Real Price): {mae_rf:,.0f}")
print(f"RMSE (Real Price): {rmse_rf:,.0f}")
print(f"R2 Score: {r2_rf:.5f}")


In [ ]:
import optuna
import xgboost as xgb
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 1000, 4000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.3),
        'max_depth': trial.suggest_int('max_depth', 2, 7),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = xgb.XGBRegressor(**param)
    
   
    scores = cross_val_score(
        model, X_train, y_train, 
        scoring='neg_mean_absolute_error', 
        cv=3,
        n_jobs=-1
    )
    
    mae = -scores.mean()
    return mae

print("Starting Optuna Study...")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50) 

print("\nBest Parameters from Optuna:")
print(study.best_params)


print("\nTraining final model with best parameters...")
best_xgb_model_optuna = xgb.XGBRegressor(**study.best_params, random_state=42, n_jobs=-1)
best_xgb_model_optuna.fit(X_train, y_train)

y_pred_xgb = best_xgb_model_optuna.predict(X_test)

mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

print("\n--- Final Model Evaluation on Test Data ---")
print(f"MAE: {mae_xgb:,.0f}")
print(f"RMSE: {rmse_xgb:,.0f}")
print(f"R2 Score: {r2_xgb:.5f}")


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor

rf_param_grid = {
    'n_estimators': [100, 300, 500, 800, 1000],
    'max_depth': [None, 10, 20, 30, 40],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.3, 0.5, 1.0]
}

rf_base = RandomForestRegressor(random_state=42, n_jobs=-1)

rf_random_search = RandomizedSearchCV(
    estimator=rf_base, 
    param_distributions=rf_param_grid, 
    n_iter=50, 
    scoring='r2', 
    cv=3, 
    verbose=2, 
    random_state=42,
    n_jobs=-1
)

print("Starting Randomized Search for Random Forest...")
rf_random_search.fit(X_train_scaled, y_train)

print("Best Parameters Found for Random Forest:")
print(rf_random_search.best_params_)


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb


param_distributions = {
    'n_estimators': [500, 1000, 2000, 3000],          
    'learning_rate': [0.01, 0.05, 0.1, 0.2],     
    'max_depth': [3, 5, 7, 9],                   
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],          
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],  
    'min_child_weight': [1, 3, 5, 7]         
}
xgb_model = xgb.XGBRegressor(random_state=42)

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_distributions,
    n_iter=120,              
    scoring='neg_mean_absolute_error',
    cv=3,                    
    verbose=2,              
    random_state=42,
    n_jobs=-1                
)
print("Starting Randomized Search... This might take a while!")
random_search.fit(X_train, y_train)

print("Best Parameters Found:")
print(random_search.best_params_)
best_xgb_model = random_search.best_estimator_
